[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/zipfs_law_oct20_2026.ipynb)

# Zipf’s law: compare texts and stop words
IPHS 200 · Tuesday, October 20, 2026

Zipf’s law describes an approximate relationship, namely that a word’s frequency tends to decrease as its rank increases. The second-ranked word typically occurs about half as often as the first, the third about a third as often, and so on. This is a general pattern of language, but real texts depart from this pattern. Consider how Zipf's law complicates word frequency as a method for interpreting text.


## 1. Setup
Run this once. The notebook downloads NLTK’s Gutenberg texts and English stop-word list.


In [ ]:
from collections import Counter
import nltk
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

for resource in ("gutenberg", "stopwords"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Could not download {resource}. Check the connection and rerun setup.")

stop_words = set(nltk.corpus.stopwords.words("english"))


## Optional: upload your own text
Run the next cell to upload one UTF-8 `.txt` file. In section 2, check **Use_Uploaded_Text**. Your upload stays available when you change the stop-word setting; rerun this upload cell to replace it.


In [ ]:
from google.colab import files

uploaded_files = files.upload()
if len(uploaded_files) != 1:
    raise ValueError("Upload one .txt file.")
filename, file_bytes = next(iter(uploaded_files.items()))
if not filename.lower().endswith(".txt"):
    raise ValueError("Choose a plain-text (.txt) file.")
try:
    candidate_text = file_bytes.decode("utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError("Save your text as UTF-8 and upload it again.") from exc
if not candidate_text.strip():
    raise ValueError("The uploaded file is empty.")
uploaded_text = candidate_text
uploaded_name = filename
print(f"Ready: {uploaded_name}")


## 2. Choose a text and run the analysis
Choose a sample from the dropdown, or check **Use_Uploaded_Text** after uploading your novel. After changing either setting, **rerun this entire cell**. 

Choices: Jane Austen’s *Emma* or *Sense and Sensibility*; Lewis Carroll’s *Alice’s Adventures in Wonderland*; G. K. Chesterton’s *The Man Who Was Thursday*; Herman Melville’s *Moby-Dick*; or Shakespeare’s *Macbeth*. These differ in length and genre, so raw counts alone are not a fair comparison.

We lowercase words and omit tokens that contain non-letter characters as a typical preprocessing choice.  A stop-word list removes many frequent grammatical words--notice how that changes the pattern, since stop words are integral to Zipf's Law patterns.


In [ ]:
Use_Uploaded_Text = False #@param {type:"boolean"}
Book_Name = "austen-emma.txt" #@param ['austen-emma.txt','austen-sense.txt','carroll-alice.txt','chesterton-thursday.txt','melville-moby_dick.txt','shakespeare-macbeth.txt']
Keep_Stopwords = True #@param {type:"boolean"}


def frequency_table(tokens, keep_stopwords=True, stop_words=None):
    words = [word.lower() for word in tokens if word.isalpha()]
    if not keep_stopwords:
        excluded = set() if stop_words is None else stop_words
        words = [word for word in words if word not in excluded]
    counts = Counter(words)  # Fresh on every call: never mix runs or books.
    if not counts:
        raise ValueError("No words remain after filtering.")
    ranked = sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))
    table = pd.DataFrame(ranked, columns=["Word", "Frequency"])
    table["Rank"] = range(1, len(table) + 1)
    table["Relative frequency"] = table["Frequency"] / len(words)
    table["Frequency × Rank"] = table["Frequency"] * table["Rank"]
    return table[["Rank", "Word", "Frequency", "Relative frequency", "Frequency × Rank"]]


if Use_Uploaded_Text:
    if "uploaded_text" not in globals():
        raise ValueError("Run the optional upload cell first, or uncheck Use_Uploaded_Text.")
    tokens = nltk.tokenize.WordPunctTokenizer().tokenize(uploaded_text)
    text_name = uploaded_name
else:
    if Book_Name not in nltk.corpus.gutenberg.fileids():
        raise ValueError("Choose one of the available Gutenberg texts.")
    tokens = nltk.corpus.gutenberg.words(Book_Name)
    text_name = Book_Name
df = frequency_table(tokens, Keep_Stopwords, stop_words)
label = f"{text_name} — stop words {'kept' if Keep_Stopwords else 'removed'}"
print(label)
print(f"Word occurrences: {df['Frequency'].sum():,}; distinct words: {len(df):,}")
display(df.head(30))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
top = df.head(30)
axes[0].bar(top["Word"], top["Relative frequency"])
axes[0].tick_params(axis="x", labelrotation=90)
axes[0].set(xlabel="Word", ylabel="Share of retained word occurrences", title="30 most frequent words")

axes[1].loglog(df["Rank"], df["Relative frequency"], label="Observed", alpha=0.8)
reference = df["Relative frequency"].iloc[0] / df["Rank"]
axes[1].loglog(df["Rank"], reference, "--", label="1/rank reference (anchored at rank 1)")
axes[1].set(xlabel="Rank (log scale)", ylabel="Relative frequency (log scale)", title="Rank–frequency distribution")
axes[1].legend()
fig.suptitle(label)
fig.tight_layout()
plt.show()


## 3. Compare experiments
1. Start with one book **keeping stop words**. What dominates the top 30? Does the rank–frequency curve roughly follow the reference? Where does it depart?
2. Choose another book, keep stop words, and rerun the analysis cell. Compare the shapes and relative frequencies, not just the raw counts. 
3. Return to your first book, **remove stop words**, and rerun. Which words become prominent? How does the curve change?
4. Connect this to Voyant: when would frequent grammatical words be informative, and when might removing them help answer a question about a novel?
